In [222]:
import importlib

import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import os

import model_functions
import utils

importlib.reload(model_functions)
importlib.reload(utils)

from model_functions import load_transition_matrix, gen_risk_matrix, gen_transition_probabilities, run_markov_cohort, \
    run_mc_sim, calculate_outcomes, plot_trace, load_costs, STAGE_ORDER, run_comparison, run_mc_sim_complex

In [207]:
STAGE_ORDER = ['F0', 'F1', 'F2', 'F3', 'F4', 'HCC', 'DC', 'LT', 'post-LT', 'Death']
# TODO Add in 3_mo_post_LT, 6_mo_post_LT, 12_mo_post_LT??

MORTALITY_COLUMNS = ['Death']
# TODO split Death into liver-related, all-cause, cardiovascular?

DIR_HRP = "./" # TODO global variable

In [216]:
cycle_denom = 4  # 13

cycle_len = 1/cycle_denom   # 13
n_cycles = 88 * cycle_denom  # 13
cohort_size = 5000  # 5
tm = load_transition_matrix('./final_baseline_transition_probs.csv', "_trial", cycle_length=cycle_len)
rm = gen_risk_matrix(regression_risk=1.56, progression_risk=0.6097424116233234)

initial_prevalence = [.628, 0.309, 0.035, 0.016, 0.012, 0,0,0,0,0]

In [ ]:
# treatment starting at 12, 72 weeks of treatment
mc_trace_12_72, drug_trace_12_72 = run_mc_sim_complex(tm, initial_prevalence, n_cycles, cohort_size, starting_age=12,
                      cycle_length=cycle_len, risk_matrix=rm,
                                          begin_tx_at_cycle=0, tx_cycle_num=6, drug_stages=['F2', 'F3'])

In [55]:
# treatment starting at 18, 72 weeks of treatment
mc_trace_18_72, drug_trace_18_72 = run_mc_sim_complex(tm, initial_prevalence, n_cycles, cohort_size, starting_age=12,
                      cycle_length=cycle_len, risk_matrix=rm,
                                          begin_tx_at_cycle=6*cycle_denom, tx_cycle_num=6, drug_stages=['F2', 'F3'])

In [57]:
((mc_trace_12_72 != 9).sum(axis=1) / 4 + 12).describe()

count    5000.000000
mean       58.224000
std        16.788666
min        12.250000
25%        46.937500
50%        60.750000
75%        71.000000
max        93.750000
dtype: float64

In [58]:
((mc_trace_18_72 != 9).sum(axis=1) / 4 + 12).describe()

count    5000.000000
mean       58.065550
std        16.933678
min        12.500000
25%        46.500000
50%        60.750000
75%        71.000000
max        91.750000
dtype: float64

In [10]:
(drug_trace_12_72.sum(axis=1)).describe()

count    1000.000000
mean       10.887000
std         8.678222
min         0.000000
25%         0.000000
50%        18.000000
75%        18.000000
max        18.000000
dtype: float64

In [11]:
drug_trace_18_72.sum(axis=1).describe()

count    1000.00000
mean       10.49600
std         8.75995
min         0.00000
25%         0.00000
50%        18.00000
75%        18.00000
max        18.00000
dtype: float64

In [59]:
import numpy as np
import pandas as pd

def convert_to_cohort_trace(markov_trace, stage_order, age_index=None):
    """
    Converts an individual-level Markov trace to a cohort-level trace.
    
    Parameters:
    - markov_trace: 2D numpy array or Pandas DataFrame (n_individuals, n_cycles) 
                    containing the integer state indices.
    - stage_order: List of state names (strings) corresponding to the indices.
    - age_index: Optional array, list, or pandas Index to use as the row labels (ages).
    
    Returns:
    - pd.DataFrame: Cohort trace (rows = age, columns = stages, values = proportions)
    """
    # Ensure we are working with a fast numpy integer array
    trace_array = np.asarray(markov_trace).astype(int)
    n_individuals, n_cols = trace_array.shape
    n_states = len(stage_order)
    
    # Pre-allocate an empty matrix for the results
    cohort_matrix = np.zeros((n_cols, n_states))
    
    # Iterate through each cycle (column) and count the states
    for t in range(n_cols):
        # bincount tallies occurrences of each state index (0, 1, 2...).
        # minlength ensures states with 0 patients are still represented.
        state_counts = np.bincount(trace_array[:, t], minlength=n_states)
        
        # Convert to proportions and store in our matrix
        cohort_matrix[t, :] = state_counts / n_individuals
        
    # Convert back to a nicely formatted Pandas DataFrame
    cohort_df = pd.DataFrame(cohort_matrix, columns=stage_order)
    cohort_df[['age_float', 'age']] = markov_trace.columns.tolist()
        
    return cohort_df

In [197]:
mc_test = convert_to_cohort_trace(mc_trace_18_72, STAGE_ORDER)
oc_18_72 = calculate_outcomes(mc_test, cycle_len, 'Semaglutide',
                     on_drug=drug_trace_18_72)

Total LY:  45.94055
Total QALY:  38.483740102774995
Total Cost:  843852.7485833231
Discounted LY:  23.885036144917436
Discounted QALY:  20.5415835426082
Discounted Cost:  385022.63627878134


In [198]:
mc_test = convert_to_cohort_trace(mc_trace_12_72, STAGE_ORDER)
oc_12_72 = calculate_outcomes(mc_test, cycle_len, 'Semaglutide',
                     on_drug=drug_trace_12_72)

Total LY:  46.099000000000004
Total QALY:  38.64992608155
Total Cost:  844622.497834692
Discounted LY:  23.955462185516147
Discounted QALY:  20.621253341082728
Discounted Cost:  384375.9972305996


In [199]:
(384375.9972305996 - 385022.63627878134) / (20.621253341082728 - 20.5415835426082)

-8116.489065659188

***This is cost saving in this run***

In [3]:
import numpy as np
import pandas as pd
import os
import concurrent.futures
import multiprocessing

# --- Worker Function for Parallel Processing ---
# This function runs a "chunk" of the total iterations on a single core.
def _sim_worker(args):
    (n_iter_chunk, base_transition_matrix, init_state, n_cycles, starting_age, 
     cycle_length, risk_matrix, begin_tx_at_cycle, tx_cycle_num, drug_stages, overall_mortality) = args

    rng = np.random.default_rng()
    
    # We assume STAGE_ORDER is defined globally in the module
    markov_trace = np.zeros((n_iter_chunk, n_cycles + 1))
    on_drug = np.zeros((n_iter_chunk, n_cycles + 1))
    num_drug_cycles = np.zeros(n_iter_chunk)

    for i in range(n_iter_chunk):
        starting_state = rng.choice(len(init_state), p=init_state)
        markov_trace[i, 0] = starting_state
        
        for t in range(n_cycles):
            age = starting_age + t * cycle_length
            
            local_risk_matrix = risk_matrix.copy() if risk_matrix is not None else None
            this_stage_idx = int(markov_trace[i, t])
            this_stage = STAGE_ORDER[this_stage_idx]

            if this_stage in MORTALITY_COLUMNS:
                markov_trace[i, t + 1:] = this_stage_idx
                break
            
            local_on_drug = (((drug_stages is not None and this_stage in drug_stages) or
                              (drug_stages is None)) and
                             begin_tx_at_cycle <= t and
                             num_drug_cycles[i] < tx_cycle_num)
            
            if local_on_drug:
                on_drug[i, t] = 1
                num_drug_cycles[i] += 1
            else:
                local_risk_matrix = None

            t_matrix = gen_transition_probabilities(
                base_transition_matrix.copy(), 
                np.floor(age),
                overall_mortality,  # Passed from memory, not disk
                local_risk_matrix
            )
            
            markov_trace[i, t + 1] = rng.choice(
                len(init_state), 
                p=t_matrix.iloc[this_stage_idx, :].values
            )

    return markov_trace, on_drug


# --- Main Parallelized Function ---
def run_mc_sim_complex(base_transition_matrix, init_state, n_cycles, n_iter, 
                       starting_age=12, cycle_length=1., risk_matrix=None, 
                       begin_tx_at_cycle=0, tx_cycle_num=None, drug_stages=None,
                       n_jobs=-1):
    
    assert len(init_state) == len(STAGE_ORDER), "Initial state vector length must match number of states."
    assert len(init_state) == len(base_transition_matrix), "Transition matrix size must match number of states."

    if tx_cycle_num is None:
        tx_cycle_num = n_cycles

    # Update gen_transition_probabilities to accept this object directly.
    overall_mortality = pd.read_csv(os.path.join(DIR_HRP, "background_mortality.csv"), index_col=0) 

    # 2. DETERMINE CPU CORES & CHUNKING STRATEGY
    if n_jobs == -1:
        n_jobs = multiprocessing.cpu_count() # Uses all 8 cores on your VM
    
    # Split the n_iter into chunks for each core to process
    base_chunk = n_iter // n_jobs
    chunks = [base_chunk] * n_jobs
    for i in range(n_iter % n_jobs):
        chunks[i] += 1 # Distribute remainder

    # Create arguments for each worker
    worker_args = [
        (chunk, base_transition_matrix, init_state, n_cycles, starting_age, 
         cycle_length, risk_matrix, begin_tx_at_cycle, tx_cycle_num, drug_stages, overall_mortality)
        for chunk in chunks if chunk > 0
    ]

    # 3. RUN IN PARALLEL
    all_markov_traces = []
    all_on_drugs = []
    
    with concurrent.futures.ProcessPoolExecutor(max_workers=n_jobs) as executor:
        results = executor.map(_sim_worker, worker_args)
        
        for trace_chunk, drug_chunk in results:
            all_markov_traces.append(trace_chunk)
            all_on_drugs.append(drug_chunk)

    # Recombine results from all cores
    final_markov_trace = np.vstack(all_markov_traces)
    final_on_drug = np.vstack(all_on_drugs)

    # 4. FORMAT DATAFRAMES
    trace_df = pd.DataFrame(final_markov_trace).T
    trace_df['age_float'] = np.linspace(starting_age, starting_age + n_cycles * cycle_length, n_cycles + 1)
    trace_df['age'] = np.floor(trace_df['age_float'])
    trace_df.set_index(['age_float', 'age'], inplace=True)

    on_drug_df = pd.DataFrame(final_on_drug).T
    on_drug_df['age_float'] = np.linspace(starting_age, starting_age + n_cycles * cycle_length, n_cycles + 1)
    on_drug_df['age'] = np.floor(on_drug_df['age_float'])
    on_drug_df.set_index(['age_float', 'age'], inplace=True)

    return trace_df.T, on_drug_df.T

# Longer treatment 

In [62]:

# treatment starting at 12, 260 weeks of treatment
mc_trace_12_260, drug_trace_12_260 = run_mc_sim_complex(tm, initial_prevalence, n_cycles, cohort_size, starting_age=12,
                      cycle_length=cycle_len, risk_matrix=rm,
                                          begin_tx_at_cycle=0, tx_cycle_num=cycle_denom*5, drug_stages=['F2', 'F3'])
# treatment starting at 18, 260 weeks of treatment
mc_trace_18_260, drug_trace_18_260 = run_mc_sim_complex(tm, initial_prevalence, n_cycles, cohort_size, starting_age=12,
                      cycle_length=cycle_len, risk_matrix=rm,
                                          begin_tx_at_cycle=cycle_denom*6, tx_cycle_num=cycle_denom*5, drug_stages=['F2', 'F3'])

In [63]:
mc_test = convert_to_cohort_trace(mc_trace_12_260, STAGE_ORDER)
oc_12_260 = calculate_outcomes(mc_test, cycle_len, 'Semaglutide',
                     on_drug=drug_trace_12_260)

Total LY:  46.2217
Total QALY:  38.81430069065
Total Cost:  844177.985071782
Discounted LY:  23.98151410713555
Discounted QALY:  20.665681282836882
Discounted Cost:  389048.5891048988


In [64]:
mc_test = convert_to_cohort_trace(mc_trace_18_260, STAGE_ORDER)
oc_18_260 = calculate_outcomes(mc_test, cycle_len, 'Semaglutide',
                     on_drug=drug_trace_18_260)

Total LY:  46.62825000000001
Total QALY:  39.13320120625
Total Cost:  841903.2528852421
Discounted LY:  24.11504356561453
Discounted QALY:  20.769872131445457
Discounted Cost:  386679.4169867566


In [201]:
(389048.5891048988 - 386679.4169867566) / (20.665681282836882 - 20.769872131445457)

-22738.773604222424

***In this run, it is strongly dominated***

# Investigate Causes of Difference

In [202]:


def get_death_stage(df):
    df_1 = df != 9
    df_2 = df
    true_counts = df_1.sum(axis=1)
    last_true_idx = true_counts.values - 1
    
    # 2. Identify the column names
    last_true_columns = df_1.columns[last_true_idx].get_level_values(0)
    
    # 3. Retrieve the values from df_2
    # We use NumPy indexing here because it is significantly faster than .lookup()
    row_indices = np.arange(len(df_2))
    last_true_values = df_2.values[row_indices, last_true_idx]
    
    # 4. Report results
    results = pd.DataFrame({
        'age_at_death': last_true_columns + 1/13,
        'from_stage': [STAGE_ORDER[int(v)] for v in last_true_values]
    }, index=df_1.index)
    return results

In [203]:
results_12_72 = get_death_stage(mc_trace_12_72)
results_18_72 = get_death_stage(mc_trace_18_72)

In [204]:
results_12_72['age_at_death'].describe()

count    5000.000000
mean       58.050923
std        16.788666
min        12.076923
25%        46.764423
50%        60.576923
75%        70.826923
max        93.576923
Name: age_at_death, dtype: float64

In [205]:
results_18_72['age_at_death'].describe()

count    5000.000000
mean       57.892473
std        16.933678
min        12.326923
25%        46.326923
50%        60.576923
75%        70.826923
max        91.576923
Name: age_at_death, dtype: float64

In [72]:
results_12_72.groupby('from_stage')['age_at_death'].describe()

,count,mean,std,min,25%,50%,75%,max
from_stage,,,,,,,,
DC,388.0,53.873315,16.449774,14.576923,41.326923,54.951923,67.139423,87.076923
F0,787.0,53.211294,18.333703,12.076923,39.451923,55.076923,67.326923,93.076923
F1,1347.0,57.893738,17.125056,14.076923,46.576923,60.576923,70.826923,93.576923
F2,1212.0,60.797426,15.870574,14.826923,51.014423,63.076923,73.326923,91.326923
F3,690.0,62.353010,14.195889,16.576923,53.826923,64.326923,73.014423,91.576923
F4,208.0,64.141827,13.920040,16.826923,57.951923,66.201923,73.889423,88.326923
HCC,299.0,49.172241,14.832019,12.326923,38.701923,51.076923,59.451923,83.576923
LT,3.0,58.326923,16.859715,43.826923,49.076923,54.326923,65.576923,76.826923
post-LT,66.0,69.129953,11.580862,39.076923,62.201923,70.201923,77.951923,89.576923


In [73]:
results_18_72.groupby('from_stage')['age_at_death'].describe()

,count,mean,std,min,25%,50%,75%,max
from_stage,,,,,,,,
DC,378.0,53.984330,15.345394,13.326923,43.576923,55.826923,65.076923,87.076923
F0,765.0,53.276923,18.709451,12.326923,38.076923,55.826923,68.076923,89.576923
F1,1316.0,57.234408,17.406926,13.076923,44.326923,59.951923,70.326923,91.076923
F2,1232.0,60.291615,15.508369,13.826923,50.576923,62.326923,72.076923,89.326923
F3,707.0,63.249837,14.354571,15.326923,55.076923,66.076923,73.576923,90.076923
F4,190.0,64.776923,13.897256,17.826923,57.139423,66.451923,76.076923,91.576923
HCC,345.0,49.516778,16.943686,13.076923,37.326923,50.326923,62.826923,87.576923
LT,1.0,27.326923,NaN,27.326923,27.326923,27.326923,27.326923,27.326923
post-LT,66.0,69.148893,12.073156,34.076923,63.889423,72.326923,76.514423,90.326923


In [ ]:
# investigate when on treatment (age and which stage**)

In [78]:
drug_trace_12_72.sum(axis=1).describe()

count    5000.000000
mean        4.529200
std         2.545357
min         0.000000
25%         4.000000
50%         6.000000
75%         6.000000
max         6.000000
dtype: float64

Slightly more people got treatment in the age 12 case. Makes sense (for anyone who died before they reached 18? Does not fully explain it...)

In [90]:
drug_trace_12_72.loc[results_12_72[results_12_72['age_at_death'] < 18].index].sum(axis=1).sum()

39.0

In [77]:
drug_trace_18_72.sum(axis=1).describe()

count    5000.000000
mean        4.485600
std         2.567238
min         0.000000
25%         3.000000
50%         6.000000
75%         6.000000
max         6.000000
dtype: float64

In [130]:
def get_stage_counts(mc_trace, drug_trace):
    stage_counts = {s: 0 for s in STAGE_ORDER}
    for i in range(len(mc_trace)):
        counts = mc_trace.iloc[i, :][drug_trace.iloc[i,:] == 1].value_counts()
        counts.index = [STAGE_ORDER[int(v)] for v in counts.index]
        for s in counts.index:
            stage_counts[s] += counts[s]
    return stage_counts

In [131]:
stage_counts_12 = get_stage_counts(mc_trace_12_72, drug_trace_12_72)
stage_counts_12

{'F0': 0,
 'F1': 0,
 'F2': 21744,
 'F3': 902,
 'F4': 0,
 'HCC': 0,
 'DC': 0,
 'LT': 0,
 'post-LT': 0,
 'Death': 0}

In [132]:
stage_counts_18 = get_stage_counts(mc_trace_18_72, drug_trace_18_72)
stage_counts_18

{'F0': 0,
 'F1': 0,
 'F2': 21017,
 'F3': 1411,
 'F4': 0,
 'HCC': 0,
 'DC': 0,
 'LT': 0,
 'post-LT': 0,
 'Death': 0}

In [158]:
tmp = np.unique(mc_trace_12_72_f3.values, return_counts=True)[1] - np.unique(mc_trace_18_72_f3.values, return_counts=True)[1]
{s: v for s, v in zip(STAGE_ORDER, tmp)}

{'F0': -946,
 'F1': 2630,
 'F2': 4266,
 'F3': -1220,
 'F4': -1546,
 'HCC': -110,
 'DC': 118,
 'LT': 1,
 'post-LT': 402,
 'Death': -3595}

## Test only F3 (to isolate the timing of the tx effect, not the impact depending on stage)

In [117]:
mc_trace_12_72_f3, drug_trace_12_72_f3 = run_mc_sim_complex(tm, initial_prevalence, n_cycles, cohort_size, starting_age=12,
                      cycle_length=cycle_len, risk_matrix=rm,
                                          begin_tx_at_cycle=0, tx_cycle_num=6, drug_stages=['F3'])

In [118]:
# treatment starting at 18, 72 weeks of treatment, only F3
mc_trace_18_72_f3, drug_trace_18_72_f3 = run_mc_sim_complex(tm, initial_prevalence, n_cycles, cohort_size, starting_age=12,
                      cycle_length=cycle_len, risk_matrix=rm,
                                          begin_tx_at_cycle=6*cycle_denom, tx_cycle_num=6, drug_stages=['F3'])

In [119]:
mc_test = convert_to_cohort_trace(mc_trace_12_72_f3, STAGE_ORDER)
oc_12_72_f3 = calculate_outcomes(mc_test, cycle_len, 'Semaglutide',
                     on_drug=drug_trace_12_72_f3)

Total LY:  46.40100000000001
Total QALY:  38.8910479097
Total Cost:  829241.7503532014
Discounted LY:  24.056253656955942
Discounted QALY:  20.695980014630404
Discounted Cost:  378374.80705489323


In [120]:
mc_test = convert_to_cohort_trace(mc_trace_18_72_f3, STAGE_ORDER)
oc_18_72_f3 = calculate_outcomes(mc_test, cycle_len, 'Semaglutide',
                     on_drug=drug_trace_18_72_f3)

Total LY:  46.22125
Total QALY:  38.718612018875
Total Cost:  829135.6324870652
Discounted LY:  23.95170645457103
Discounted QALY:  20.59959854163776
Discounted Cost:  376682.0495120345


In [121]:
# $/QALY
(378374.80705489323 - 376682.0495120345) / (20.695980014630404 - 20.59959854163776)

17563.10098091073

In [133]:
stage_counts_12_f3 = get_stage_counts(mc_trace_12_72_f3, drug_trace_12_72_f3)
stage_counts_12_f3

{'F0': 0,
 'F1': 0,
 'F2': 0,
 'F3': 13790,
 'F4': 0,
 'HCC': 0,
 'DC': 0,
 'LT': 0,
 'post-LT': 0,
 'Death': 0}

In [134]:
stage_counts_18_f3 = get_stage_counts(mc_trace_18_72_f3, drug_trace_18_72_f3)
stage_counts_18_f3

{'F0': 0,
 'F1': 0,
 'F2': 0,
 'F3': 13629,
 'F4': 0,
 'HCC': 0,
 'DC': 0,
 'LT': 0,
 'post-LT': 0,
 'Death': 0}

In [135]:
results_12_72_f3 = get_death_stage(mc_trace_12_72_f3)
results_18_72_f3 = get_death_stage(mc_trace_18_72_f3)

In [136]:
results_12_72_f3['age_at_death'].describe()

count    5000.000000
mean       58.352923
std        16.684263
min        12.576923
25%        47.264423
50%        60.826923
75%        71.076923
max        92.076923
Name: age_at_death, dtype: float64

In [137]:
results_18_72_f3['age_at_death'].describe()

count    5000.000000
mean       58.173173
std        16.954834
min        12.076923
25%        47.326923
50%        60.826923
75%        71.076923
max        92.326923
Name: age_at_death, dtype: float64

In [138]:
results_12_72_f3.groupby('from_stage')['age_at_death'].describe()

,count,mean,std,min,25%,50%,75%,max
from_stage,,,,,,,,
DC,358.0,52.470080,16.287517,13.076923,41.326923,52.701923,64.764423,89.076923
F0,771.0,54.207922,18.856508,12.576923,39.451923,58.076923,68.826923,91.076923
F1,1285.0,57.795211,16.659299,12.826923,46.576923,59.826923,70.576923,90.326923
F2,1290.0,60.739714,15.647663,14.576923,50.576923,63.326923,72.576923,90.576923
F3,737.0,63.226855,14.315825,18.826923,54.826923,64.576923,74.576923,90.076923
F4,189.0,65.572955,13.275479,12.826923,58.326923,67.076923,75.576923,92.076923
HCC,303.0,50.161081,15.515411,12.826923,40.201923,50.826923,61.201923,89.826923
LT,1.0,68.826923,NaN,68.826923,68.826923,68.826923,68.826923,68.826923
post-LT,66.0,65.239802,12.465837,25.076923,59.326923,65.326923,73.201923,88.076923


In [139]:
results_18_72_f3.groupby('from_stage')['age_at_death'].describe()

,count,mean,std,min,25%,50%,75%,max
from_stage,,,,,,,,
DC,368.0,54.303146,16.652900,12.576923,43.576923,56.326923,66.639423,89.076923
F0,798.0,53.434066,19.274123,12.576923,38.389423,56.326923,69.014423,90.076923
F1,1248.0,57.434896,17.447651,12.076923,45.576923,60.451923,71.076923,90.576923
F2,1270.0,60.894836,15.260332,15.326923,51.326923,62.951923,72.326923,90.076923
F3,740.0,62.441450,14.516275,18.826923,54.326923,64.576923,73.326923,92.076923
F4,195.0,64.298718,13.299624,12.826923,57.451923,66.326923,73.201923,88.076923
HCC,315.0,49.927717,15.717195,15.076923,38.701923,51.326923,60.576923,85.826923
LT,1.0,72.826923,NaN,72.826923,72.826923,72.826923,72.826923,72.826923
post-LT,65.0,72.026923,11.196592,37.326923,67.826923,73.576923,79.826923,92.326923


In [158]:
tmp = np.unique(mc_trace_12_72_f3.values, return_counts=True)[1] - np.unique(mc_trace_18_72_f3.values, return_counts=True)[1]
{s: v for s, v in zip(STAGE_ORDER, tmp)}

{'F0': -946,
 'F1': 2630,
 'F2': 4266,
 'F3': -1220,
 'F4': -1546,
 'HCC': -110,
 'DC': 118,
 'LT': 1,
 'post-LT': 402,
 'Death': -3595}

No meaningful difference in number of liver transplants (1 addl) but a lot less HCC, F3, F4 and a substantial amount of more life (0.18 years on avg).

In [178]:
def get_age_at_drug_start(drug_trace):
    age_l = np.zeros(len(drug_trace))
    for i in range(len(drug_trace)):
        age = drug_trace.columns[(drug_trace.iloc[i] == 1)]
        if len(age) > 0:
            age_l[i] = age[0][0]
    return age_l

In [180]:
drug_start_12_72_f3 = get_age_at_drug_start(drug_trace_12_72_f3)

In [182]:
(drug_start_12_72_f3 != 0).sum()

2427

In [194]:
drug_start_12_72_f3[(drug_start_12_72_f3 != 0)].mean()

39.165945611866505

In [189]:
tmp = drug_start_12_72_f3[(drug_start_12_72_f3 < 18) & (drug_start_12_72_f3 != 0)]
len(tmp)

230

In [190]:
tmp.mean()

14.1

In [191]:
drug_start_18_72_f3 = get_age_at_drug_start(drug_trace_18_72_f3)

In [195]:
drug_start_18_72_f3[(drug_start_18_72_f3 != 0)].mean()

40.294650291423814

In [192]:
(drug_start_18_72_f3 != 0).sum()

2402

# Cohort starting at F2/F3

In [217]:
sum_f2_f3 = 0.035 + 0.016
initial_prevalence = [0,0, 0.035/sum_f2_f3, 0.016/sum_f2_f3, 0, 0,0,0,0,0]

In [218]:
# treatment starting at 12, 72 weeks of treatment
mc_trace_12_72_star, drug_trace_12_72_star = run_mc_sim_complex(tm, initial_prevalence, n_cycles, cohort_size, starting_age=12,
                      cycle_length=cycle_len, risk_matrix=rm,
                                          begin_tx_at_cycle=0, tx_cycle_num=6, drug_stages=['F2', 'F3'])

# treatment starting at 18, 72 weeks of treatment
mc_trace_18_72_star, drug_trace_18_72_star = run_mc_sim_complex(tm, initial_prevalence, n_cycles, cohort_size, starting_age=12,
                      cycle_length=cycle_len, risk_matrix=rm,
                                          begin_tx_at_cycle=6*cycle_denom, tx_cycle_num=6, drug_stages=['F2', 'F3'])

In [219]:
mc_test = convert_to_cohort_trace(mc_trace_12_72_star, STAGE_ORDER)
oc_12_72_star = calculate_outcomes(mc_test, cycle_len, 'Semaglutide',
                     on_drug=drug_trace_12_72_star)

Total LY:  39.78105
Total QALY:  32.561046520675
Total Cost:  968920.4947453123
Discounted LY:  21.701642758042034
Discounted QALY:  18.12601592240813
Discounted Cost:  484875.9540114651


In [220]:
mc_test = convert_to_cohort_trace(mc_trace_18_72_star, STAGE_ORDER)
oc_18_72_star = calculate_outcomes(mc_test, cycle_len, 'Semaglutide',
                     on_drug=drug_trace_18_72_star)

Total LY:  39.83415
Total QALY:  32.52771640545
Total Cost:  986550.9216090058
Discounted LY:  21.638691210722754
Discounted QALY:  18.02216571228197
Discounted Cost:  495082.616584028


In [223]:
from utils import gen_cea
basic_results = gen_cea({'12': {'QALY': oc_12_72_star[0].sum(), 'Cost': oc_12_72_star[1].sum()}, 
                         '18': {'QALY': oc_18_72_star[0].sum(), 'Cost': oc_18_72_star[1].sum()}})
basic_results

,QALY,Cost,Delta_Cost,Delta_QALY,ICER
18,18.022166,495082.616584,495082.616584,18.022166,27470.761533
12,18.126016,484875.954011,-10206.662573,0.103850,-98282.541366
